# 02_add_dedup_sheets — 중복 정리 시트 2개 추가

**한 줄 요약:** 01에서 만든 엑셀에 시트 2개를 더한다. ① `same_dedup_keepdiff`(같은 물질+같은 IC50는 1줄로 합치고, IC50가 다르면 남김) ② `median_per_compound`(물질당 1줄, IC50는 중앙값).
**큰 흐름:** ① 데이터 읽기·함수 준비 → ② 시트1 만들기 → ③ 시트2 만들기 → ④ 엑셀에 시트 추가

> **📌 이 노트북 읽는 법**: 각 코드 셀은 **[① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기]** 순서.
> ③은 그 셀에 **처음 나온** 함수·문법 설명(기초 반복은 *(01에서 설명)* 으로 생략). `# ...`=주석.

### 준비 — 폴더 위치 맞추기
노트북을 어느 폴더에서 열어도 프로젝트 최상위에서 실행되도록 이동한다(그래야 `data/...` 경로가 맞음).

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
while not os.path.isdir('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')  # data/ 폴더를 찾을 때까지 상위로 (하위 폴더에서 열어도 동작)
print('작업 폴더:', os.getcwd())

🔎 **코드 뜯어보기** *(01에서 설명)*: `os.chdir('..')`=상위 폴더로 이동, `os.path.isdir`=폴더 존재 확인, `print`=화면 출력.

### 셀 1 — 준비 + 데이터 읽기
루트로 이동하고, 01에서 만든 엑셀의 `all_data` 시트를 표로 읽는다.

In [ ]:
import numpy as np
import pandas as pd

OUT = "data/HSD17B13_IC50_merged.xlsx"

df = pd.read_excel(OUT, sheet_name="all_data")

🔎 **코드 뜯어보기 (셀 1)** *(import/chdir는 01에서 설명)*
- `pd.read_excel(OUT, sheet_name="all_data")` : 엑셀에서 **특정 시트**를 골라 표로 읽기.

### 셀 2 — 유효 행만 남기고 비교키·집계 함수 준비
SMILES·IC50가 있는 행만 남기고, IC50 반올림 비교키를 만들고, 집계에 쓸 작은 함수 2개를 정의한다.

In [ ]:
d = df.dropna(subset=["canonical_smiles", "ic50_nM"]).copy()
d["ic50_key"] = d["ic50_nM"].round(3)   # 부동소수 잡음 방지용 비교키


def join_sources(s):
    return ", ".join(sorted(s.dropna().astype(str).unique()))


def first_valid(s):
    v = s.dropna()
    return v.iloc[0] if len(v) else np.nan

🔎 **코드 뜯어보기 (셀 2)**
- `df.dropna(subset=[열들])` : 지정한 열에 빈 값이 있는 행을 **버림**. `.copy()`=복사.
- `d["ic50_nM"].round(3)` : 소수 3자리로 반올림 → 부동소수 미세오차로 '같은 값'이 달라 보이는 것 방지(비교용 키).
- `def join_sources(s): return ", ".join(sorted(s.dropna().astype(str).unique()))` : 함수. `.unique()`=중복 없는 값들, `sorted`=정렬, `", ".join(...)`=쉼표로 이어 하나의 글자로.
- `def first_valid(s): ... s.iloc[0] ...` : 빈 값 뺀 뒤 **첫 번째 값**을 반환(`iloc[0]`=0번째).

### 셀 3 — 시트1: 같은 IC50는 합치고 다른 IC50는 남기기
(물질, IC50) 조합 단위로 묶어 대표값을 뽑는다 → 같은 값은 1줄, 다른 값은 별도 줄.

In [ ]:
s1 = (d.groupby(["canonical_smiles", "ic50_key"], as_index=False)
        .agg(compound_id=("compound_id", first_valid),
             smiles=("smiles", first_valid),
             ic50_nM=("ic50_nM", "first"),
             relation=("relation", first_valid),
             pChEMBL=("pChEMBL", first_valid),
             sources=("source", join_sources),
             n_records=("source", "size")))

🔎 **코드 뜯어보기 (셀 3)**
- `d.groupby(["canonical_smiles", "ic50_key"], as_index=False)` : 두 열의 **조합 단위로 묶기**(같은 물질+같은 IC50끼리). `as_index=False`=묶은 키를 일반 열로 둠.
- `.agg(새열=("원본열", 함수))` : **이름 있는 집계** — 각 그룹에서 원본열에 함수를 적용해 새 열 만들기. `"first"`=첫 값, `"size"`=개수, 우리가 만든 `first_valid`/`join_sources`도 사용.

### 셀 4 — 시트1: 물질별 IC50 종류 수 세고 정렬
한 물질에 IC50 값이 몇 종류 남았는지 세고(=아직 남은 중복 줄), 보기 좋게 정렬·열 정리.

In [ ]:
s1["n_ic50_values"] = s1.groupby("canonical_smiles")["ic50_nM"].transform("size")
s1 = s1.sort_values(["n_ic50_values", "canonical_smiles", "ic50_nM"],
                    ascending=[False, True, True]).reset_index(drop=True)
s1 = s1.drop(columns=["ic50_key"], errors="ignore")
s1 = s1[["canonical_smiles", "compound_id", "smiles", "ic50_nM", "relation",
         "pChEMBL", "sources", "n_records", "n_ic50_values"]]

🔎 **코드 뜯어보기 (셀 4)**
- `.groupby("...")["ic50_nM"].transform("size")` : 그룹의 크기(줄 수)를 계산해 **원래 각 행에 되돌려** 붙임(=이 물질의 IC50 종류가 몇 개?).
- `sort_values([열들], ascending=[...])` : 여러 열로 정렬. `.drop(columns=[...], errors="ignore")`=열 제거. `df[[열 순서]]`=열 재배치.

### 셀 5 — 시트2: 물질당 1줄, IC50 중앙값
물질 단위로 묶어 IC50의 중앙값·최소·최대·측정수를 계산한다.

In [ ]:
s2 = (d.groupby("canonical_smiles", as_index=False)
        .agg(compound_id=("compound_id", first_valid),
             smiles=("smiles", first_valid),
             ic50_nM_median=("ic50_nM", "median"),
             ic50_nM_min=("ic50_nM", "min"),
             ic50_nM_max=("ic50_nM", "max"),
             n_measurements=("ic50_nM", "size"),
             sources=("source", join_sources)))

🔎 **코드 뜯어보기 (셀 5)**
- `.agg(... median=("ic50_nM", "median"), ...)` : 그룹별 **중앙값·최소·최대·개수**를 한 번에. `"median"`=중앙값(값들을 정렬했을 때 가운데).

### 셀 6 — 시트2: 부등호 섞임 표시 후 정렬
한 물질에 `>`·`<` 값이 섞였는지 표시(중앙값 해석 주의용)하고 정렬한다.

In [ ]:
qual = (d.assign(q=d["relation"].astype(str).str.strip().isin([">", "<", ">=", "<="]))
          .groupby("canonical_smiles")["q"].any().reset_index(name="has_qualifier"))
s2 = s2.merge(qual, on="canonical_smiles", how="left")
s2 = s2.sort_values(["n_measurements", "canonical_smiles"],
                    ascending=[False, True]).reset_index(drop=True)
s2 = s2[["canonical_smiles", "compound_id", "smiles", "ic50_nM_median",
         "ic50_nM_min", "ic50_nM_max", "n_measurements", "has_qualifier", "sources"]]

🔎 **코드 뜯어보기 (셀 6)**
- `d.assign(q=조건)` : 임시 새 열 `q`(부등호가 섞였는지 True/False)를 추가한 표를 만듦. `.isin([">","<",...])`=값이 목록에 있는지.
- `.groupby("...")["q"].any()` : 그룹에 True가 **하나라도** 있으면 True. `.reset_index(name="has_qualifier")`=결과를 열 이름 붙여 표로.
- `s2.merge(qual, on="canonical_smiles", how="left")` : 두 표를 **공통 열(on) 기준으로 붙이기**(엑셀 VLOOKUP과 비슷). `how="left"`=왼쪽 표 기준.

### 셀 7 — 기존 엑셀에 시트 2개 추가 저장
기존 시트/서식은 보존하면서 새 시트 두 개를 덧붙여 저장하고 요약을 출력한다.

In [ ]:
with pd.ExcelWriter(OUT, engine="openpyxl", mode="a",
                    if_sheet_exists="replace") as w:
    s1.to_excel(w, sheet_name="same_dedup_keepdiff", index=False)
    s2.to_excel(w, sheet_name="median_per_compound", index=False)

print("추가 완료 →", OUT)
print(f"[시트1 same_dedup_keepdiff] {len(s1)}행 "
      f"(입력 {len(d)}행에서 같은 IC50 중복 제거)")
print(f"  - IC50 값이 2개 이상 남은 물질 행: "
      f"{int((s1['n_ic50_values'] > 1).sum())}")
print(f"[시트2 median_per_compound] {len(s2)}행 (물질당 1줄)")
print(f"  - 측정 2건 이상인 물질: {int((s2['n_measurements'] > 1).sum())}")
print(f"  - 부등호 섞인 물질: {int(s2['has_qualifier'].sum())}")

🔎 **코드 뜯어보기 (셀 7)**
- `pd.ExcelWriter(OUT, mode="a", if_sheet_exists="replace")` : **mode="a"**=기존 파일에 **덧붙이기**(기존 시트 보존). `if_sheet_exists="replace"`=같은 이름 시트가 있으면 교체.
- `f"...{len(s1)}..."` : f-문자열(값을 글자에 끼워 넣기). `(s1['n_ic50_values']>1).sum()`=조건 참인 행 수.